# Noise Functions (Simplex, Worley)

A self-contained refresher on the **two noise functions you reach for after Perlin**:
**Simplex noise** (Ken Perlin's 2001 redesign of gradient noise) and **Worley noise**
(Steven Worley's 1996 *cellular* noise). They solve different problems — Simplex is a
cheaper, artifact-free *smooth* noise; Worley is *structured* noise built from distances
to feature points.

For the gradient-noise fundamentals (lattice, gradients, fBm/octaves), see the companion
notebook [`perlin-noise.ipynb`](./perlin-noise.ipynb). This one assumes you know those and
focuses on what's *different*.

**Domain:** Procedural Generation  ·  **runnable:** yes

## 1. What & Why

Procedural content needs noise: a function that turns coordinates into "random but coherent"
values. Perlin noise is the classic, but it has two warts — its cost grows as **2^n** with
dimension (it samples every corner of a hypercube), and its square lattice leaves faint
**axis-aligned directional artifacts**. Two functions fix or sidestep this:

- **Simplex noise** (Perlin, 2001) replaces the square/cube grid with a **simplex** grid —
  triangles in 2D, tetrahedra in 3D, the simplest shape that tiles each dimension. A point
  falls inside one simplex with only **n+1** corners (3 in 2D vs. 4 for Perlin), so cost is
  **O(n^2)** instead of O(2^n), the gradient is well-defined everywhere, and there's **no
  directional bias**. Use it as a drop-in replacement for Perlin whenever you want smooth
  noise, especially in 3D/4D (animated noise) where Perlin's cost explodes.

- **Worley noise** (Worley, 1996), a.k.a. **cellular / Voronoi noise**, is a totally
  different idea: scatter random **feature points** through space; at each location return a
  function of the **distance to the nearest** point(s). This produces **organic cellular
  structure** — stone, cracked mud, scales, leather, water caustics, cell walls — patterns
  that smooth gradient noise simply cannot make.

**Reach for Simplex** when you'd use Perlin but want it faster / cleaner / higher-dimensional.
**Reach for Worley** when you want *structure with edges and cells*, not soft clouds. They
compose: Worley for the cell layout, Simplex layered on top for surface detail.

> **Patent footnote:** Perlin's *Simplex* algorithm in 3D+ was patented (expired
> January 2022). The widely-used **OpenSimplex** / **OpenSimplex2** noise is a patent-free
> alternative with similar properties; most modern engines ship OpenSimplex2.

## 2. Mental Model

**Simplex — tile the plane with triangles instead of squares.**
Perlin asks "which grid *square* am I in?" and blends its 4 corners. Simplex skews space so
the square grid becomes a grid of triangles, asks "which *triangle* am I in?", and sums a
**radial bump** from each of the 3 corner gradients. Fewer corners means cheaper; triangles
have no preferred axis means no grid artifacts. The magic is the **skew/unskew** transform
that lets you find the containing simplex with plain `floor()`.

**Worley — drop pebbles, measure to the nearest.**
Imagine pebbles scattered on a floor. Stand anywhere and measure the distance to the
**closest** pebble (call it **F1**). Plot that distance as brightness and you get smooth
bumps that rise away from each pebble — **cells**. Measure to the *second* closest (**F2**)
and take **F2 - F1**, and you light up exactly the places equidistant from two pebbles: the
**cell borders** (a Voronoi diagram). Swap Euclidean distance for Manhattan or Chebyshev and
the cells change shape from round blobs to diamonds to squares.

## 3. Key Concepts

**Simplex**
- **Simplex** — the simplest polytope that tiles a dimension: triangle (2D), tetrahedron
  (3D). A point lies in exactly one, defined by **n+1** corners.
- **Skew / unskew (F, G factors)** — `F2 = (sqrt(3)-1)/2`, `G2 = (3-sqrt(3))/6`. Skewing by F
  maps the triangular lattice to an integer square lattice so you can `floor()` to find the
  cell; G unskews contributions back. These constants are *the* fiddly part of every
  implementation.
- **Contribution & radial falloff** — each corner contributes `(r^2 - d^2)^4 * (gradient . offset)`
  where `r^2 = 0.5`. The `(...)^4` is a finite-support kernel: a corner only affects points
  inside its circle, so a sample touches just the n+1 nearby corners.
- **Gradient table + permutation** — same trick as Perlin: a small set of gradient vectors
  indexed through a hashed permutation table for cheap, repeatable pseudo-randomness.

**Worley**
- **Feature points** — random points, usually one (or a few) per grid cell so you can search
  only the 3x3 neighborhood instead of all points.
- **F1, F2, ... Fn** — distance to the 1st, 2nd, ... nearest feature point. `F1` gives cells;
  `F2-F1` gives edges/cracks; `F2` and combinations give varied textures.
- **Distance metric** — Euclidean (round cells), Manhattan / L1 (diamonds), Chebyshev / Linf
  (squares). Changing it completely restyles the output.
- **Jitter** — how far feature points move from cell centers (0 = regular grid, 1 = fully
  random). Controls regularity.

**Shared with all noise:** **fBm / octaves** (sum scaled copies for fractal detail),
**seamless tiling** (wrap coordinates / feature points), and a **seed** for reproducibility.

## 4. Setup

Both functions are a few array operations — **NumPy alone** runs everything here, and
`matplotlib` just visualizes. No GPU, no large downloads, no API keys.

```bash
pip install numpy matplotlib
```

For production you'd typically use a maintained library rather than hand-rolling:

- **`opensimplex`** — pure-Python OpenSimplex (patent-free Simplex variant). The cells below
  use it *if installed* and fall back to a from-scratch implementation otherwise.
- **`pyfastnoiselite`** / **FastNoiseLite** — fast C noise (Simplex, Worley/cellular,
  Perlin, fBm) with bindings for many languages; the go-to for real-time use.

In [ ]:
# Core deps — NumPy does the math, matplotlib renders. Uncomment to install.
# %pip install numpy matplotlib
import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless-safe; remove for inline figures in JupyterLab
import matplotlib.pyplot as plt

SEED = 7
print("NumPy", np.__version__)

def show(imgs, titles, cmap="viridis"):
    """Tiny helper: render a row of 2D arrays."""
    n = len(imgs)
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.2))
    axes = np.atleast_1d(axes)
    for ax, im, t in zip(axes, imgs, titles):
        ax.imshow(im, cmap=cmap, origin="lower")
        ax.set_title(t, fontsize=9)
        ax.axis("off")
    fig.tight_layout()
    return fig

print("setup OK")

## 5. Worked Examples

Four small examples, all CPU-friendly on a 256x256 grid:
1. **Worley F1** — cellular noise from scratch.
2. **Worley F2-F1 & distance metrics** — Voronoi edges and how the metric restyles cells.
3. **Simplex noise** from scratch (vectorized 2D), plus fBm octaves.
4. **Using the `opensimplex` library** — the production call shape, gated behind an import check.

### Example 1 — Worley (cellular) noise from scratch: F1 distance

Scatter one jittered feature point per grid cell, then for every pixel measure the distance
to the nearest feature point in the surrounding 3x3 cells. That nearest distance is **F1**.

In [ ]:
def worley(size=256, cells=8, jitter=1.0, metric="euclidean", which=(0,), seed=SEED):
    """Return Worley F-values. `which` picks which sorted neighbors (0=F1, 1=F2, ...)."""
    rng = np.random.default_rng(seed)
    # One random feature point per cell, placed at cell origin + jittered offset.
    off_x = rng.random((cells, cells)) * jitter
    off_y = rng.random((cells, cells)) * jitter

    # Pixel coordinates in "cell units" (0..cells across the image).
    gx, gy = np.meshgrid(np.linspace(0, cells, size, endpoint=False),
                         np.linspace(0, cells, size, endpoint=False))
    ci = np.floor(gx).astype(int)
    cj = np.floor(gy).astype(int)

    best = np.full((size, size, max(which) + 1), np.inf)
    for dj in (-1, 0, 1):                 # search the 3x3 neighborhood of cells
        for di in (-1, 0, 1):
            ni, nj = ci + di, cj + dj      # neighbor cell coords (may go out of range)
            wi, wj = ni % cells, nj % cells  # toroidal wrap to look up the feature point
            # Place the point back at the *unwrapped* cell so distances stay continuous.
            px = ni + off_x[wj, wi]
            py = nj + off_y[wj, wi]
            dx, dy = gx - px, gy - py
            if metric == "euclidean":
                d = np.hypot(dx, dy)
            elif metric == "manhattan":
                d = np.abs(dx) + np.abs(dy)
            elif metric == "chebyshev":
                d = np.maximum(np.abs(dx), np.abs(dy))
            else:
                raise ValueError(metric)
            # keep the smallest distances per pixel (insertion into sorted top-k)
            stacked = np.concatenate([best, d[..., None]], axis=-1)
            best = np.sort(stacked, axis=-1)[..., : max(which) + 1]
    return [best[..., k] for k in which]

f1, = worley(which=(0,))
f1 = f1 / f1.max()  # normalize to [0, 1]
print("F1 cellular noise:", f1.shape,
      f"min={f1.min():.3f} max={f1.max():.3f} mean={f1.mean():.3f}")
show([f1], ["Worley F1 (cells)"], cmap="bone").savefig("/tmp/worley_f1.png", dpi=70)
print("saved /tmp/worley_f1.png")

### Example 2 — `F2 - F1` edges and how the distance metric restyles cells

`F1` alone gives soft blobs. The combination **`F2 - F1`** is near-zero everywhere except on
the boundaries equidistant from two feature points — i.e. it draws the **Voronoi edges /
cracks**. Swapping the distance metric changes cell shape entirely.

In [ ]:
# Voronoi cracks: bright where two feature points are equidistant.
f1, f2 = worley(which=(0, 1))
edges = (f2 - f1)
edges = edges / edges.max()
print(f"F2-F1 edges: mean={edges.mean():.3f}  (low = inside cells, high = borders)")

# Same F1 noise under three distance metrics -> round / diamond / square cells.
metrics = ["euclidean", "manhattan", "chebyshev"]
imgs, titles = [], []
for m in metrics:
    (g1,) = worley(cells=6, metric=m, which=(0,))
    imgs.append(g1 / g1.max())
    titles.append(f"F1 — {m}")
fig = show(imgs, titles, cmap="bone")
fig.savefig("/tmp/worley_metrics.png", dpi=70)
show([edges], ["F2 - F1 (Voronoi edges)"], cmap="magma").savefig("/tmp/worley_edges.png", dpi=70)
shapes = {"euclidean": "round blobs", "manhattan": "diamonds", "chebyshev": "squares"}
for m in metrics:
    print(f"  metric {m:>9}: cells render as {shapes[m]}")

### Example 3 — Simplex noise from scratch (vectorized 2D) + fBm

A compact, vectorized 2D simplex implementation following Stefan Gustavson's reference. The
skew constants `F2`/`G2` find the containing triangle with `floor()`; each of the 3 corners
contributes a radial bump `(0.5 - d^2)^4 * (grad . offset)`. Stacking octaves gives fractal
terrain — exactly like Perlin fBm, but artifact-free.

In [ ]:
# 8 gradient directions; indexed through a hashed permutation table.
_GRAD = np.array([[1,1],[-1,1],[1,-1],[-1,-1],[1,0],[-1,0],[0,1],[0,-1]], dtype=float)

def _perm(seed=SEED):
    p = np.random.default_rng(seed).permutation(256)
    return np.concatenate([p, p])  # doubled to avoid index wrap

def simplex2(x, y, perm):
    F2 = 0.5 * (np.sqrt(3.0) - 1.0)
    G2 = (3.0 - np.sqrt(3.0)) / 6.0
    s = (x + y) * F2                       # skew input space to the simplex grid
    i = np.floor(x + s).astype(int)
    j = np.floor(y + s).astype(int)
    t = (i + j) * G2
    x0, y0 = x - (i - t), y - (j - t)      # offset from cell origin (unskewed)

    # Which of the two triangles? Lower or upper.
    i1 = (x0 > y0).astype(int)
    j1 = 1 - i1
    x1, y1 = x0 - i1 + G2,       y0 - j1 + G2
    x2, y2 = x0 - 1.0 + 2*G2,    y0 - 1.0 + 2*G2

    ii, jj = i & 255, j & 255
    g0 = _GRAD[perm[ii      + perm[jj     ]] % 8]
    g1 = _GRAD[perm[ii + i1 + perm[jj + j1]] % 8]
    g2 = _GRAD[perm[ii + 1  + perm[jj + 1 ]] % 8]

    def bump(dx, dy, g):
        tt = 0.5 - dx*dx - dy*dy           # finite-support radial kernel
        contrib = (tt**4) * (g[..., 0]*dx + g[..., 1]*dy)
        return np.where(tt > 0, contrib, 0.0)

    n = bump(x0, y0, g0) + bump(x1, y1, g1) + bump(x2, y2, g2)
    return 70.0 * n                        # scale roughly into [-1, 1]

def field(size=256, scale=6.0, seed=SEED):
    perm = _perm(seed)
    xs = np.linspace(0, scale, size)
    X, Y = np.meshgrid(xs, xs)
    return simplex2(X, Y, perm)

base = field()
print(f"single-octave simplex: min={base.min():.3f} max={base.max():.3f} "
      f"mean={base.mean():.3f}  (~[-1, 1], zero-centered)")

# fBm: sum octaves at doubling frequency, halving amplitude.
def fbm(size=256, octaves=5, scale=4.0, persistence=0.5, seed=SEED):
    out = np.zeros((size, size)); amp, freq, norm = 1.0, 1.0, 0.0
    for o in range(octaves):
        out += amp * field(size, scale * freq, seed=seed + o)
        norm += amp; amp *= persistence; freq *= 2.0
    return out / norm

terrain = fbm()
show([base, terrain], ["simplex (1 octave)", "simplex fBm (5 octaves)"]).savefig(
    "/tmp/simplex.png", dpi=70)
print(f"5-octave fBm terrain: min={terrain.min():.3f} max={terrain.max():.3f}")

### Example 4 — Using the `opensimplex` library (production path)

Hand-rolling is great for understanding; in real code use a maintained, patent-free library.
This cell uses **`opensimplex`** if it's importable and otherwise prints the call shape and
falls back — so the notebook executes either way (the same pattern you'd use to gate an
API-key/large-download cell behind `os.getenv`).

In [ ]:
import importlib.util

if importlib.util.find_spec("opensimplex") is not None:
    import opensimplex
    opensimplex.seed(SEED)
    xs = np.linspace(0, 6, 256)
    # noise2array is vectorized: grid = outer product of the x and y axes.
    grid = opensimplex.noise2array(xs, xs)
    print(f"opensimplex.noise2array -> {grid.shape}, "
          f"min={grid.min():.3f} max={grid.max():.3f}")
    show([grid], ["opensimplex.noise2array"]).savefig("/tmp/opensimplex.png", dpi=70)
else:
    print("opensimplex not installed — `pip install opensimplex` to run this cell.")
    print("Call shape would be:")
    print("    import opensimplex")
    print("    opensimplex.seed(7)")
    print("    grid = opensimplex.noise2array(np.linspace(0, 6, 256),")
    print("                                   np.linspace(0, 6, 256))  # (256, 256)")
    # Fall back to our from-scratch simplex so downstream cells still have data.
    grid = field()
    print(f"fallback from-scratch field -> {grid.shape}, "
          f"min={grid.min():.3f} max={grid.max():.3f}")

## 6. Gotchas & Pitfalls

- **Output range is not a clean +/-1.** Like Perlin, raw simplex rarely hits its theoretical
  extremes — the `70.0` scale constant is empirical and dimension-specific (different for 3D).
  **Normalize from the actual `min`/`max`** before mapping to colors or heights, don't assume
  `[-1, 1]`.
- **Worley F-values aren't normalized either.** F1's max depends on grid density and jitter.
  Always rescale, and remember `F1` grows *away* from points — invert it (`1 - F1`) if you
  want bright cell centers.
- **Naive Worley is O(points^2).** Comparing every pixel to every feature point is deadly at
  scale. Use the **grid trick**: one point per cell, search only the 3x3 neighborhood — what
  the example does. Without it, large textures crawl.
- **Forgetting toroidal wrap breaks tiling.** Worley cells along the edges won't match unless
  feature points wrap; simplex tiles only if you sample a periodic coordinate range. Seamless
  textures need explicit wrapping, not luck.
- **Simplex skew constants are easy to get subtly wrong.** A flipped sign in `i1/j1` or a wrong
  `G2` gives noise that *looks* plausible but has artifacts. Validate against a reference
  (mean ~ 0, smooth, no grid lines) — or just use a library.
- **Distance-metric surprise.** Manhattan/Chebyshev Worley reintroduces axis alignment (the
  very thing Simplex avoids). Fine for stylized looks, wrong if you wanted isotropy.
- **Patent confusion.** Perlin's original *Simplex* (3D+) was patented until 2022; if your
  dependency audit flags it, switch to **OpenSimplex2** — same look, no encumbrance.

## 7. When to Use vs Alternatives

| Function | Best for | Trade-off |
|---|---|---|
| **Simplex / OpenSimplex** | Smooth clouds, terrain, animated 3D/4D noise; a faster, artifact-free Perlin | More complex to implement; range needs empirical scaling |
| **Worley / cellular** | Cells, cracks, scales, stone, caustics — *structured* patterns | Cost scales with feature points (mitigate with the grid trick); not "soft" |
| **Perlin** ([notebook](./perlin-noise.ipynb)) | The classic; tons of reference code; fine in 2D | 2^n cost, faint axis artifacts — Simplex supersedes it |
| **Value noise** | Cheapest smooth noise, trivial to code | Blockier, less natural than gradient noise |
| **Voronoi/Delaunay** ([notebook](./voronoi-delaunay.ipynb)) | Exact cell *polygons*, region adjacency, meshes | Combinatorial structure, not a cheap per-pixel field like Worley |
| **Diamond-Square** ([notebook](./diamond-square.ipynb)) | Quick fractal heightmaps on a grid | Grid-bound, classic creasing artifacts |

**Rules of thumb:** want soft/cloudy → **Simplex** (Perlin if you just need quick reference
code). Want cellular/edged structure → **Worley**. Need the actual polygon cells, not a field
→ **Voronoi/Delaunay**. And you rarely use one alone: a common recipe is **Worley for
macro-structure x Simplex fBm for micro-detail**, multiplied or blended together.

## 8. Resources

- **Stefan Gustavson — "Simplex noise demystified" (2005)** — the canonical, readable
  derivation with reference C code (the basis of Example 3):
  https://weber.itn.liu.se/~stegu/simplexnoise/simplexnoise.pdf
- **Ken Perlin — "Improving Noise" (SIGGRAPH 2002) / Simplex slides**:
  https://mrl.cs.nyu.edu/~perlin/paper445.pdf
- **Steven Worley — "A Cellular Texture Basis Function" (SIGGRAPH 1996)** — the original
  Worley/cellular noise paper: https://www.rhythmiccanvas.com/research/papers/worley.pdf
- **OpenSimplex2** — patent-free Simplex-style noise, the modern default (multi-language
  implementations): https://github.com/KdotJPG/OpenSimplex2
- **`opensimplex` (PyPI)** — the pure-Python library used in Example 4:
  https://github.com/lmas/opensimplex
- **FastNoiseLite** — fast C noise (Simplex, cellular/Worley, fractal) for real-time use:
  https://github.com/Auburn/FastNoiseLite
- **The Book of Shaders — Cellular Noise** — interactive, visual intro to Worley:
  https://thebookofshaders.com/12/
- **Red Blob Games — Making maps with noise functions** — practical fBm/terrain recipes:
  https://www.redblobgames.com/maps/terrain-from-noise/